# Quantum chemistry on Amazon Braket with Qrunch

You are going to calculate how much energy it takes to strip an electron off an ammonia molecule, using
a variational quantum algorithm, and you are going to run the same calculation on two very different
machines.

| Where it runs | What it is | Roughly how long |
|---|---|---|
| Your notebook | Kvantify's local statevector simulator, exact maths | a few minutes |
| Amazon Braket SV1 | Managed cloud statevector simulator, 34 qubits | longer — every step is a service call |

The chemistry, the problem definition and the algorithm settings stay identical across both. Two lines
change. That is the point of the lab: the same code, moved onto managed cloud infrastructure.


:::alert{type=info}
Adapted from [Kvantify/qrunch_tutorials](https://github.com/Kvantify/qrunch_tutorials),
used under the MIT License. Copyright (c) 2025 Kvantify. The chemistry is Kvantify's work.
:::

## Setup

Qrunch is already installed on this kernel. Run the cell below to check which version you have.


In [ ]:
import importlib.metadata as md

print("Qrunch", md.version("qrunch"))


Now register your licence.

Drag `license.txt` from the Kvantify registration email into the `workshop/` folder in the file browser on the
left, then run the cell below. If you get a file-not-found error, the file is somewhere else in the tree:
check the file browser and move it next to this notebook.


In [ ]:
import qrunch as qc

qc.register_license_file("license.txt")
print("Licence registered")


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

output_path = Path("output")
output_path.mkdir(exist_ok=True, parents=True)


## The molecule

Ionization energy is the difference between two calculations: the neutral molecule, and the same molecule
with one electron removed. So you build two configurations from the same geometry, one with charge 0 and
one with charge +1. The cation has an unpaired electron, hence `spin_difference=1`.

`cc-pvdz` is the basis set, the set of mathematical functions used to describe the electron orbitals.
Bigger basis sets are more accurate and much more expensive.


In [ ]:
structure_path = Path("data/nh3_eq.xyz")
basis_set = "cc-pvdz"

neutral_configuration = qc.build_molecular_configuration(
    molecule=structure_path,
    basis_set=basis_set,
    charge=0,
    spin_difference=0,
)

cation_configuration = qc.build_molecular_configuration(
    molecule=structure_path,
    basis_set=basis_set,
    charge=1,
    spin_difference=1,
)


## The active space

Simulating every electron in ammonia on today's quantum hardware is out of the question. Instead you pick an
active space: the handful of orbitals near the frontier where the interesting chemistry happens. Everything
below stays frozen.

Six orbitals and four alpha electrons gives a circuit of about 12 qubits, which real hardware can handle.
This trade-off between accuracy and qubit count is the central practical decision in quantum chemistry today.


In [ ]:
problem_builder_creator = (
    qc.problem_builder_creator()
    .ground_state()
    .standard()
    .add_problem_modifier()
    .active_space(6, 4)
)

problem_builder_neutral = problem_builder_creator.create()
problem_builder_cation = problem_builder_creator.create()

problem_neutral = problem_builder_neutral.build_unrestricted(neutral_configuration)
problem_cation = problem_builder_cation.build_unrestricted(cation_configuration)

print(f"Neutral: {problem_neutral.electron_configuration.number_of_alpha_electrons} alpha, "
      f"{problem_neutral.electron_configuration.number_of_beta_electrons} beta electrons")
print(f"Cation:  {problem_cation.electron_configuration.number_of_alpha_electrons} alpha, "
      f"{problem_cation.electron_configuration.number_of_beta_electrons} beta electrons")


---
## Part 1: local simulator

Kvantify's excitation-gate simulator runs on this notebook instance. Nothing leaves the machine, nothing is
billed, and it computes exact statevectors rather than sampling. That makes it the reference SV1 gets
compared against.

FAST-VQE builds its circuit one gate at a time: each macro-iteration it works out which excitation gate
would lower the energy most, adds it, and re-optimises.

Ammonia's measured ionization energy is about 0.37 Ha, roughly 10 eV. A 12-qubit active space cannot
reproduce that — it throws away most of the electron correlation, so expect to land below the experimental
value. How far below is the interesting question.


In [ ]:
sampler_local = qc.sampler_creator().excitation_gate().create()
estimator_local = qc.estimator_creator().excitation_gate().create()

gate_selector_local = (
    qc.gate_selector_creator().fast().with_sampler(sampler_local).with_shots(None).create()
)

adaptive_options = qc.options.IterativeVqeOptions(
    max_iterations=50,
    force_all_iterations=True,
)

calculator_local = (
    qc.calculator_creator()
    .vqe()
    .iterative()
    .standard()
    .with_estimator(estimator=estimator_local)
    .with_options(options=adaptive_options)
    .with_gate_selector(gate_selector=gate_selector_local)
    .create()
)


In [ ]:
%%time
result_neutral_local = calculator_local.calculate(problem_neutral)
result_cation_local = calculator_local.calculate(problem_cation)


In [ ]:
energies_neutral_local = np.array(result_neutral_local.total_energy_per_macro_iteration.values)
energies_cation_local = np.array(result_cation_local.total_energy_per_macro_iteration.values)
ionization_local = energies_cation_local - energies_neutral_local

plt.figure(figsize=(8, 5))
plt.plot(ionization_local, marker="o", linestyle="--", color="yellowgreen", label="Local simulator")
plt.xlabel("Iteration")
plt.ylabel("Ionization energy [Ha]")
plt.title("$NH_3$ ionization energy, local simulation")
plt.grid()
plt.legend()
plt.show()

print(f"Final ionization energy: {ionization_local[-1]:.6f} Ha")


Two questions worth sitting with. Is the curve converged, or still drifting downward? And how big is the
gap to the experimental 0.37 Ha?

That gap is active-space truncation, not a bug. You froze all but six orbitals to get the problem down to
12 qubits, and truncation error is the price. Widening the active space closes it and costs qubits — which
is the central practical trade-off in quantum chemistry today.


---
## Part 2: Amazon Braket SV1

Same molecule, same active space, same algorithm, same number of iterations. The calculation now runs on
SV1, Braket's managed statevector simulator, instead of on this instance.

Exactly two things change:

- the sampler points at the Braket backend rather than the local one
- the gate selector uses 1000 shots instead of exact expectation values, because a device returns
  measurement counts, not amplitudes

That second change is the interesting one. SV1 does the same exact linear algebra as the local simulator, so
any difference you see comes from shot noise in the gate selection step, not from the simulator being less
accurate.

On cost: SV1 is billed at $0.075 per minute of simulation time with a three-second minimum per task, and a
12-qubit circuit finishes well inside that minimum. So the bill is set by how many tasks the algorithm
submits, not by how hard each circuit is. The Braket free tier covers the first hour of simulation per
month. You will measure the real task count and the real cost at the end of this part rather than take our
word for it.


In [ ]:
from braket.devices import Devices
from braket.tracking import Tracker

sampler_sv1 = (
    qc.sampler_creator()
    .backend()
    .choose_backend()
    .amazon_braket(device=Devices.Amazon.SV1)
    .create()
)

estimator_sv1 = qc.estimator_creator().excitation_gate().create()

gate_selector_sv1 = (
    qc.gate_selector_creator().fast().with_sampler(sampler_sv1).with_shots(1000).create()
)

calculator_sv1 = (
    qc.calculator_creator()
    .vqe()
    .iterative()
    .standard()
    .with_estimator(estimator=estimator_sv1)
    .with_options(options=adaptive_options)
    .with_gate_selector(gate_selector=gate_selector_sv1)
    .create()
)


The run below is wrapped in the Braket SDK's cost tracker, which records every task it submits. This is
much slower than Part 1 — the arithmetic is trivial, but every step is a round trip to the service. Start
it and read on.


In [ ]:
%%time
with Tracker() as sv1_tracker:
    result_neutral_sv1 = calculator_sv1.calculate(problem_neutral)
    result_cation_sv1 = calculator_sv1.calculate(problem_cation)


In [ ]:
energies_neutral_sv1 = np.array(result_neutral_sv1.total_energy_per_macro_iteration.values)
energies_cation_sv1 = np.array(result_cation_sv1.total_energy_per_macro_iteration.values)
ionization_sv1 = energies_cation_sv1 - energies_neutral_sv1

plt.figure(figsize=(8, 5))
plt.plot(ionization_local, marker="o", linestyle="--", color="yellowgreen", label="Local simulator")
plt.plot(ionization_sv1, marker="s", linestyle="-", color="steelblue", label="Braket SV1")
plt.xlabel("Iteration")
plt.ylabel("Ionization energy [Ha]")
plt.title("$NH_3$ ionization energy, local vs Braket SV1")
plt.grid()
plt.legend()
plt.show()

print(f"Local final: {ionization_local[-1]:.6f} Ha")
print(f"SV1 final:   {ionization_sv1[-1]:.6f} Ha")
print(f"Difference:  {abs(ionization_sv1[-1] - ionization_local[-1]):.8f} Ha")


### What that run actually cost

`Tracker` reports the tasks it saw, the simulation time Braket billed for, and the price. Compare the billed
duration against the free tier's one hour of simulation per month.


In [ ]:
FREE_TIER_MINUTES = 60  # Braket free tier: 1 hour of on-demand simulation per month

for device_arn, stats in sv1_tracker.quantum_tasks_statistics().items():
    print(device_arn.split("/")[-1])
    print(f"  tasks:            {sum(stats['tasks'].values())}  {stats['tasks']}")
    print(f"  shots:            {stats['shots']}")

    billed_duration = stats.get("billed_execution_duration")
    if billed_duration is None:
        print("  billed time:      not reported for this device")
    else:
        billed = billed_duration.total_seconds() / 60
        print(f"  billed time:      {billed:.2f} min "
              f"({billed / FREE_TIER_MINUTES:.1%} of the monthly free hour)")

print(f"\nSimulator cost:     ${sv1_tracker.simulator_tasks_cost():.2f} before free tier")


Two things to take from those numbers. Check how the task count compares to the number of iterations you
asked for — the gate selection step samples the device more than once per iteration, and that is what
drives the bill. And note that almost every task will have hit the three-second billing floor, so cost
tracks the number of calls rather than the difficulty of each circuit.

Compare the total against the free tier's hour of simulation per month. That headroom is why cloud
simulators are the sensible place to develop and validate a quantum algorithm.


---
## Your turn

Have a play. These all run locally, so they cost nothing.

1. Push `max_iterations` up and re-run Part 1. Does the energy keep falling, or was 50 enough?
2. Widen the active space to `active_space(8, 6)` and re-run. What happens to the runtime, and to the answer?
3. Ammonia in water behaves differently from ammonia in a vacuum. The cell below rebuilds the neutral
   configuration inside a polarizable continuum with the dielectric constant of water. Build the cation the
   same way, re-run, and see how much the ionization energy shifts.


In [ ]:
from qrunch.chemistry.molecule.solvent import PCMMethod, SolventPCM

neutral_configuration_water = qc.build_molecular_configuration(
    molecule=structure_path,
    basis_set=basis_set,
    charge=0,
    spin_difference=0,
    solvent=SolventPCM(
        dielectric_constant=78.4,
        method=PCMMethod.IEF_PCM,
    ),
)


---
## What you did

| Backend | Type | Noise | What it cost |
|---|---|---|---|
| Kvantify `excitation_gate` | Local statevector, exact | None | Nothing |
| Amazon Braket SV1 | Managed cloud statevector | Shot noise only | Whatever the tracker reported, inside the free tier |

One codebase, two backends, two lines different between them. Develop against the local simulator because
it is instant and free, validate on SV1 because it scales past what one machine can hold, and spend money on
real hardware only when the question genuinely needs it.

Running on a QPU is the same change again: point the sampler at a hardware device instead of SV1. The
economics are what differ — hardware is billed per task plus per shot, and current devices are noisy enough
that the answer needs error mitigation before it means anything. See
[Amazon Braket pricing](https://aws.amazon.com/braket/pricing/) and the device list in the Braket console.

The honest state of the field: for ammonia, classical chemistry still wins comfortably. What you have built
is the workflow, so that when hardware crosses over, the code is already portable.
